# VLADE · AVA-VLA / OpenVLA-OFT 단일 추론 점검

이 노트북은 공개 체크포인트 하나를 로드해 전면 카메라·손목 카메라 이미지와 8차원 proprio 입력으로 **8×7 액션 청크를 한 번 생성**합니다. LIBERO+ 성공률 평가나 FFT 학습이 아닙니다. LIBERO+ 데이터셋과 평가 assets는 사용하지 않습니다.

2026-10-04에 Drive 이관 후 **NVIDIA A100-SXM4-40GB (39.5 GiB)**에서 두 모델의 설치·모델 로드·단일 추론을 완료했습니다. 두 모델 모두 **유한한 8×7 액션 생성** 검사를 통과했습니다.

| 모델 | 결과 | PyTorch 최대 할당 GPU 메모리 |
| --- | --- | --- |
| OpenVLA-OFT | passed · 8×7 · NaN/Inf 없음 | 14.97 GiB |
| AVA-VLA | passed · 8×7 · NaN/Inf 없음 | 14.82 GiB |

검사 조건은 검은색 256×256 RGB 이미지 2장, 0으로 채운 8차원 상태, `pick up the object`, seed 7, batch 1, BF16입니다. warm-up 없이 모델별 한 번 실행했으며, 저장된 소요 시간은 모델 로드와 추론을 합한 시간입니다.

결과 JSON과 패키지 버전 기록은 Drive와 로컬의 `experiments/runs/inference_smoke/`에 보관했습니다.

- `openvla_oft_20261004T055453Z.json`
- `ava_vla_20261004T060317Z.json`

검사 후 런타임 연결 해제 및 삭제를 완료했고, 세션 관리에서 **활성 세션 없음**을 확인했습니다. LIBERO(+) 성공률, FFT forward/backward/save/load, AVA의 다음 시점 temporal-context 동작은 이번 검사에 포함하지 않았습니다.

모델별로 서로 다른 Transformers 포크가 필요합니다. `MODEL`을 선택하면 `/content`에 해당 모델 전용 Python 3.11 환경을 만듭니다. 두 모델을 검사하려면 첫 실행 결과를 보관한 뒤 `MODEL`을 바꾸어 셀을 다시 실행하십시오. 설치 셀은 인터넷에서 Python과 패키지를 내려받고, 추론 셀은 **로컬/Drive 체크포인트만 사용**합니다.


## 1. A100 및 VLADE 경로 지정

Colab에서 GPU 유형을 A100으로 선택하십시오. Drive의 공유 폴더가 `MyDrive`에 보이지 않으면 해당 폴더의 바로가기를 내 드라이브에 추가하거나, 아래 `PROJECT_ROOT`를 실제 마운트 경로로 바꾸십시오. `PROJECT_ROOT`는 `third_party/`, `checkpoints/`, `experiments/`를 포함하는 **VLADE 루트**여야 합니다.


In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
MODEL = 'openvla_oft'  # 'openvla_oft' 또는 'ava_vla'
PROJECT_ROOT = Path('/content/drive/MyDrive/colab/VLADE/VLADE')
if not (PROJECT_ROOT / 'third_party').is_dir() and (PROJECT_ROOT / 'VLADE/third_party').is_dir():
    PROJECT_ROOT = PROJECT_ROOT / 'VLADE'
COPY_LARGE_FILES_TO_SCRATCH = False  # Drive 파일 링크 읽기가 실패할 때만 True
TASK_DESCRIPTION = 'pick up the object'
UNNORM_KEY = 'libero_spatial_no_noops'

MODEL_PATHS = {
    'openvla_oft': (
        'third_party/openvla_oft',
        'checkpoints/baselines/openvla_oft/openvla-7b-oft-finetuned-libero-spatial-object-goal-10',
    ),
    'ava_vla': (
        'third_party/ava_vla',
        'checkpoints/teacher/avavla-libero-4in1',
    ),
}
assert MODEL in MODEL_PATHS, f'지원하지 않는 모델: {MODEL}'
PROJECT_ROOT = PROJECT_ROOT.expanduser().resolve()
SOURCE = PROJECT_ROOT / MODEL_PATHS[MODEL][0]
CHECKPOINT = PROJECT_ROOT / MODEL_PATHS[MODEL][1]
print('model:', MODEL, '\nproject:', PROJECT_ROOT, '\nsource:', SOURCE, '\ncheckpoint:', CHECKPOINT)


In [ ]:
import json
import torch

assert SOURCE.is_dir(), f'원본 소스가 없습니다: {SOURCE}'
assert CHECKPOINT.is_dir(), f'체크포인트가 없습니다: {CHECKPOINT}'
assert (SOURCE / 'experiments/robot/openvla_utils.py').is_file()
required = ['config.json', 'model.safetensors.index.json', 'dataset_statistics.json',
            'tokenizer.model', 'processor_config.json']
for filename in required:
    assert (CHECKPOINT / filename).is_file(), f'필수 파일 없음: {filename}'
index = json.loads((CHECKPOINT / 'model.safetensors.index.json').read_text(encoding='utf-8'))
for filename in set(index['weight_map'].values()):
    assert (CHECKPOINT / filename).is_file(), f'모델 샤드 없음: {filename}'
for pattern in ['action_head--*_checkpoint.pt', 'proprio_projector--*_checkpoint.pt']:
    assert len(list(CHECKPOINT.glob(pattern))) == 1, f'보조 가중치 누락/중복: {pattern}'
if MODEL == 'ava_vla':
    assert len(list(CHECKPOINT.glob('vision_attn_weight_generator--*_checkpoint.pt'))) == 1
stats = json.loads((CHECKPOINT / 'dataset_statistics.json').read_text(encoding='utf-8'))
assert UNNORM_KEY in stats, f'정규화 키가 없습니다: {UNNORM_KEY}'
assert torch.cuda.is_available(), 'CUDA GPU가 필요합니다.'
gpu_name = torch.cuda.get_device_name(0)
assert 'A100' in gpu_name, f'A100을 선택하십시오. 현재 GPU: {gpu_name}'
assert torch.cuda.is_bf16_supported(), 'BF16 지원 GPU가 필요합니다.'
vram_gib = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f'사전 점검 통과: {gpu_name}, {vram_gib:.1f} GiB VRAM, {len(index["weight_map"])} tensors')


## 2. 모델별 격리 환경 설치

원본 `pyproject.toml`의 고정 의존성을 Python 3.11 환경에 설치합니다. 현재 Colab 기본 Python 버전과 독립적입니다. 첫 실행에는 수 GB의 패키지 다운로드와 시간이 필요할 수 있습니다. 설치 실패 시 추론 셀을 실행하지 말고 오류를 확인하십시오. 서로 다른 두 원본을 하나의 Python 환경에 섞지 않습니다. [Colab 과거 런타임 안내](https://research.google.com/colaboratory/runtime-version-faq.html) · [uv Python 관리](https://docs.astral.sh/uv/concepts/python-versions/)


In [ ]:
import shutil
import subprocess
import sys

def run_logged(args, check=True):
    process = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, cwd=SOURCE)
    for line in process.stdout:
        print(line, end='', flush=True)
    exit_code = process.wait()
    if exit_code:
        raise RuntimeError(f'환경 설치/확인 실패 (exit={exit_code}): {args}')


VENV = Path('/content') / f'vlade_inference_{MODEL}_py311'
VENV_PYTHON = VENV / 'bin/python'
INSTALL_STAMP = VENV / 'vlade_install_ok.txt'
if shutil.which('uv') is None:
    run_logged([sys.executable, '-m', 'pip', 'install', 'uv'])
uv = shutil.which('uv')
assert uv is not None, 'uv 설치 후 다시 실행하십시오.'
if not VENV_PYTHON.is_file():
    run_logged([uv, 'venv', '--python', '3.11', str(VENV)])
if not INSTALL_STAMP.is_file():
    run_logged([uv, 'pip', 'install', '--python', str(VENV_PYTHON), '-e', str(SOURCE)])
    INSTALL_STAMP.write_text(str(SOURCE), encoding='utf-8')
else:
    assert INSTALL_STAMP.read_text(encoding='utf-8') == str(SOURCE), '환경의 모델 소스가 다릅니다.'
run_logged([str(VENV_PYTHON), '-c',
                'import torch, transformers; assert torch.cuda.is_available(); print(torch.__version__, transformers.__version__, torch.cuda.get_device_name(0))'],
               check=True)

# TFDS 4.9.3 / TensorFlow 2.15의 protobuf 호환성 유지
run_logged([uv, 'pip', 'install', '--python', str(VENV_PYTHON), 'tensorflow-metadata==1.15.0'])
run_logged([str(VENV_PYTHON), '-c', 'import tensorflow_datasets, tensorflow_metadata; from experiments.robot import openvla_utils; print("VLADE_IMPORT_OK")'])


## 3. 체크포인트 로드 및 액션 한 번 생성

원본 로더는 로컬 체크포인트의 설정·모델 코드 파일을 동기화하며 수정할 수 있습니다. Drive 원본을 보호하기 위해 `/content`에 작은 설정 파일만 복사하고 큰 가중치 파일은 심볼릭 링크로 연결한 임시 체크포인트를 사용합니다. Drive 링크 읽기에 문제가 생기면 `COPY_LARGE_FILES_TO_SCRATCH=True`로 바꿀 수 있으나 모델당 약 15GB의 추가 임시 공간이 필요합니다. 추론에서는 Hub 다운로드를 차단합니다. 검은색 합성 이미지이므로 액션 값에는 로봇 성능상의 의미가 없고, 형태·유한값·오류 유무만 확인합니다. AVA-VLA는 에피소드 첫 시점과 동일하게 temporal context 없이 시작합니다.


In [ ]:
import os
import tempfile

INFERENCE_PROGRAM = r'''
import json, os, sys, time
from types import SimpleNamespace
import numpy as np
import torch
import transformers
import tensorflow as tf
tf.config.set_visible_devices([], 'GPU')
from experiments.robot import openvla_utils as utils

selected = os.environ['VLADE_MODEL']
cfg = SimpleNamespace(
    model_family='openvla', pretrained_checkpoint=os.environ['VLADE_CHECKPOINT'],
    use_l1_regression=True, use_diffusion=False, use_film=False,
    num_images_in_input=2, use_proprio=True, center_crop=True,
    num_open_loop_steps=8, load_in_8bit=False, load_in_4bit=False,
    unnorm_key=os.environ['VLADE_UNNORM_KEY'], lora_rank=32,
)
if selected == 'ava_vla':
    cfg.multi_frame = SimpleNamespace(
        temporal_strategy='attn_weight', temporal_feature_source='action',
        temporal_feature_layer=-2, attn_weight_force_eager_attn=True,
        attn_weight_extra_layer_ids='(i for i in range(0, 32))',
        attn_weight_head_type='softmaxscore',
        attn_weight_score_config='(1.9, 0.1, 0.0)',
        attn_weight_sink_ids=[68, 75, 180, 187, 324, 331, 436, 443],
        attn_weight_sink_weight=1.0, attn_weight_attn_ratio=None,
    )
np.random.seed(7)
torch.manual_seed(7)
started = time.monotonic()
torch.set_grad_enabled(False)
vla = utils.get_vla(cfg)
processor = utils.get_processor(cfg)
action_head = utils.get_action_head(cfg, llm_dim=vla.llm_dim)
proprio_projector = utils.get_proprio_projector(cfg, llm_dim=vla.llm_dim, proprio_dim=8)
observation = {
    'full_image': np.zeros((256, 256, 3), dtype=np.uint8),
    'wrist_image': np.zeros((256, 256, 3), dtype=np.uint8),
    'state': np.zeros(8, dtype=np.float32),
}
if selected == 'ava_vla':
    generator = utils.get_vision_attn_weight_generator(
        cfg, vla.llm_dim, int(np.sqrt(vla.vision_backbone.get_num_patches()))
    )
    actions, *_ = utils.get_vla_action(
        cfg, vla, processor, observation, os.environ['VLADE_TASK_DESCRIPTION'],
        action_head=action_head, proprio_projector=proprio_projector,
        temporal_context=None, vision_attn_weight_generator=generator,
    )
else:
    actions = utils.get_vla_action(
        cfg, vla, processor, observation, os.environ['VLADE_TASK_DESCRIPTION'],
        action_head=action_head, proprio_projector=proprio_projector,
    )
array = np.asarray(actions, dtype=np.float32)
assert array.shape == (8, 7), f'예상한 액션 형태 (8, 7)가 아닙니다: {array.shape}'
assert np.isfinite(array).all(), '액션에 NaN 또는 Inf가 있습니다.'
print('VLADE_SMOKE_RESULT=' + json.dumps({
    'model': selected, 'status': 'passed', 'shape': list(array.shape),
    'actions': array.tolist(), 'load_and_inference_seconds': round(time.monotonic() - started, 2),
    'gpu': torch.cuda.get_device_name(0),
    'max_cuda_memory_gib': round(torch.cuda.max_memory_allocated() / 2**30, 2),
    'torch': torch.__version__, 'transformers': transformers.__version__,
    'python': sys.version.split()[0], 'seed': 7,
    'cuda': torch.version.cuda, 'dtype': 'bfloat16',
    'tensorflow_preprocessing_device': 'CPU',
    'batch_size': 1, 'inference_calls': 1, 'warmup_calls': 0,
    'timing_kind': 'cold_load_plus_inference_wall_time_not_latency_benchmark',
}), flush=True)
'''

result_data = None
with tempfile.TemporaryDirectory(prefix=f'vlade_{MODEL}_', dir='/content') as scratch:
    staged = Path(scratch) / 'checkpoint'
    staged.mkdir()
    for original in CHECKPOINT.iterdir():
        if not original.is_file():
            continue
        target = staged / original.name
        if original.suffix in {'.safetensors', '.pt', '.model'} and not COPY_LARGE_FILES_TO_SCRATCH:
            target.symlink_to(original.resolve())
        else:
            shutil.copy2(original, target)
    env = os.environ.copy()
    env.update({
        'VLADE_MODEL': MODEL, 'VLADE_CHECKPOINT': str(staged),
        'VLADE_UNNORM_KEY': UNNORM_KEY, 'VLADE_TASK_DESCRIPTION': TASK_DESCRIPTION,
        'HF_HUB_OFFLINE': '1', 'TRANSFORMERS_OFFLINE': '1',
        'TOKENIZERS_PARALLELISM': 'false', 'HF_HOME': '/content/vlade_hf_cache',
    })
    process = subprocess.Popen(
        [str(VENV_PYTHON), '-u', '-c', INFERENCE_PROGRAM],
        cwd=SOURCE, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    for line in process.stdout:
        print(line, end='')
        if line.startswith('VLADE_SMOKE_RESULT='):
            result_data = json.loads(line.split('=', 1)[1])
    exit_code = process.wait()
    if exit_code != 0:
        raise RuntimeError(f'{MODEL} 추론 프로세스 실패 (exit={exit_code}). 위 로그를 확인하십시오.')
assert result_data is not None, '추론 결과 표식이 없습니다.'
print('단일 액션 예측 통과:', MODEL, result_data['shape'])


In [ ]:
from datetime import datetime, timezone

assert result_data is not None and result_data['status'] == 'passed'
timestamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
manifest_path = PROJECT_ROOT / 'experiments/manifests' / ('ava_vla_source.json' if MODEL == 'ava_vla' else 'openvla_oft_source.json')
source_commit = json.loads(manifest_path.read_text(encoding='utf-8'))['source']['commit']
experiment_config = json.loads((PROJECT_ROOT / 'configs/experiments/libero_plus_fft.json').read_text(encoding='utf-8'))
checkpoint_revision = experiment_config['initial_checkpoints'][MODEL]['revision']
record = {
    'kind': 'synthetic_single_observation_smoke_test',
    'not_a_benchmark_success_rate': True,
    'timestamp_utc': timestamp,
    'repository_commit': '82480532420340d57e39529a7416991c78ce2744',
    'notebook_has_uncommitted_runtime_compatibility_fixes': True,
    'source_commit': source_commit, 'checkpoint_revision': checkpoint_revision,
    'checkpoint_path': str(CHECKPOINT.relative_to(PROJECT_ROOT)),
    'normalization_key': UNNORM_KEY,
    'task_description': TASK_DESCRIPTION, 'seed': 7,
    'dataset_revision': None, 'benchmark': None,
    'input': {'image_shape': [256, 256, 3], 'two_black_images': True, 'proprio_shape': [8]},
    'result': result_data,
    'package_freeze': subprocess.check_output([uv, 'pip', 'freeze', '--python', str(VENV_PYTHON)], text=True).splitlines(),
}
output_dir = PROJECT_ROOT / 'experiments/runs/inference_smoke'
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f'{MODEL}_{timestamp}.json'
output_path.write_text(json.dumps(record, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print('결과 기록:', output_path)

print('VLADE_SMOKE_RECORD=' + json.dumps(record, ensure_ascii=False))


## 해석과 다음 단계

`passed`는 체크포인트·프로세서·보조 헤드가 로드되고 합성 관측 한 번에서 유한한 8×7 액션이 생성되었다는 뜻만 갖습니다. LIBERO 또는 LIBERO+ 성공률, 실제 로봇 안전성, FFT 가능 여부를 증명하지 않습니다. AVA의 첫 시점 추론은 이전 행동 문맥이 없는 경로이므로 recurrent 동작 검증도 아닙니다. 실패 로그는 해당 모델의 격리 환경, Transformers 포크, GPU 메모리, 체크포인트 경로부터 확인하십시오.
